In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
!curl https://apiv2.senso.ai/api/v1/org/me \
  -H 'X-API-Key: $SENSO_FULL_KEY'

In [ ]:
# list content types
!curl https://apiv2.senso.ai/api/v1/org/content-types \
  -H "X-API-Key: $SENSO_FULL_KEY"

In [ ]:
import os, requests

KEY  = os.environ["SENSO_FULL_KEY"]
BASE = "https://apiv2.senso.ai/api/v1"
HEADERS = {"X-API-Key": KEY, "Content-Type": "application/json"}

resp = requests.post(f"{BASE}/org/kb/folders", headers=HEADERS, json={
    "name": "score_tests",
})
folder = resp.json()
folder_id = folder["kb_node_id"]
print(f"Created folder: {folder['name']} ({folder_id})")


resp = requests.post(f"{BASE}/org/kb/folders", headers=HEADERS, json={
    "name": "baseline_xml",
    "parent_id": folder_id,
})
print(f"Created subfolder: {resp.json()['name']}")

resp = requests.post(f"{BASE}/org/kb/folders", headers=HEADERS, json={
    "name": "baseline_pdf",
    "parent_id": folder_id,
})
print(f"Created subfolder: {resp.json()['name']}")

In [ ]:
# Search for a folder id
resp = requests.get(f"{BASE}/org/kb/find", headers=HEADERS, params={"q": "baseline_xml"})
for item in resp.json()["nodes"]:
    print(f"  {item['kb_node_id']} - {item['name']} ({item['type']})")

In [ ]:
import hashlib, requests

FILE_NAME = "holst-the-planets-op-32.txt"
FILE_PATH = "../data/raw_txt/" + FILE_NAME

file_bytes = open(FILE_PATH, "rb").read()

resp = requests.post(f"{BASE}/org/kb/upload", headers=HEADERS, json={
    "kb_folder_node_id": "fe8e23d9-9817-4705-9b1a-e4fbef08b237",
    "files": [{
        "filename":         FILE_NAME,
        "file_size_bytes":  len(file_bytes),
        "content_type":     "text/plain",
        "content_hash_md5": hashlib.md5(file_bytes).hexdigest(),
    }]
})
result = resp.json()["results"][0]
print(result)

requests.put(result["upload_url"], data=file_bytes)
print(f"Uploaded to {folder_id}: {result['content_id']}")